THERE ARE 3 TYPES OF RECOMMENDER SYSTEM 
1. CONTENT BASED SYSTEM
2. COLLABORATIVE LEARNING
3. HYBRID

WE ARE MAKING A CONTENT BASED COLLABORATIVE MODEL FOR MOVIE RECOMMENDER SYSTEM

STEPS TO DO IN THIS PROJECT

DATA CLEANING
EDA
TEXT PREPROCESSING
MODEL BUILDING
EVALUATION
IMPROVMENTS (depending on evaluation)
WEBSITE BUILDING
DEPLOYMENT OF THE WEBSITE ON HERUKO

In [7]:
import numpy as np
import pandas as pd

In [9]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [11]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [13]:
movies.shape

(4803, 20)

In [15]:
credits.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [17]:
#instead of having 2 datasets we merge the movies with credits with title column 
movies = movies.merge(credits , on = 'title')

In [19]:
movies.shape

(4809, 23)

In [21]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

In [23]:
#now after merging we had 23 columns where some columns are useless so we discard some columns from the dataset
#here i am writing columns which i'm going to consider in my dataset for building my model
#movie_id
#title
#overview
#genre
#cast
#crew
#keywords these i will consider rest will be removed from the dataset

In [25]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [27]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [29]:
#in our final dataset we that is the movies dataset we will be having the columns like movie_id , title , and tags (these tags will consists of keywords , genres , overview , cast and crew)
#what we will do we take overview and merge genres with it and for this we need to arrange the data first and for cast we will take top 3 names from it and from crew we will take only the directors name merge it with the other columns

#for this we need to do 2 things we will do data preprocessing like we will format this all tags columns we are going to merge and secondly we fill the missing values and remove the duplicate data from it

In [31]:
movies.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [33]:
movies.dropna(inplace=True)

In [35]:
movies.duplicated().sum()

0

In [37]:
movies.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [39]:
#this module is used to convert the string into list so we convert the genre string to list for further data preprocessing and we also convert keywords too
import ast
ast.literal_eval('[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]')

[{'id': 28, 'name': 'Action'},
 {'id': 12, 'name': 'Adventure'},
 {'id': 14, 'name': 'Fantasy'},
 {'id': 878, 'name': 'Science Fiction'}]

In [41]:
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name']) 
    return L 


In [43]:
movies['genres'] = movies['genres'].apply(convert)

In [45]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [47]:
movies['keywords'] = movies['keywords'].apply(convert)
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [48]:
#just like we did for genre and keywords we are going to convert cast as well because we want top 3 cast only so we convert the top3 cast string into list
def convert3(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 3:
            L.append(i['name'])
        counter+=1
    return L 


In [51]:
movies['cast'] = movies['cast'].apply(lambda x:x[0:3])

In [53]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[{""","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[{""","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[{""","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[{""","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[{""","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [55]:
#now we convert our crew column from which we only want director name and nothing else so we convert that string into list and extract the firector from it
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L 

In [57]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [58]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[{""",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[{""",[Gore Verbinski]
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[{""",[Sam Mendes]
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[{""",[Christopher Nolan]
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[{""",[Andrew Stanton]


In [59]:
#now comes the overview column which is also a string and we convert it into a list so here each row is converted into a string
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [60]:
movies.sample(5)

,movie_id,title,overview,genres,keywords,cast,crew
1200,1579,Apocalypto,"[Set, in, the, Mayan, civilization,, when, a, ...","[Action, Adventure, Drama, Thriller]","[loss of family, solar eclipse, slavery, jagua...","[{""",[Mel Gibson]
1103,68734,Argo,"[As, the, Iranian, revolution, reaches, a, boi...","[Drama, Thriller]","[cia, wife husband relationship, document, rev...","[{""",[Ben Affleck]
2635,13853,The Clan of the Cave Bear,"[Natural, changes, have, the, clans, moving., ...","[Adventure, Drama]","[stone age, tribe, cavemen, prehistoric advent...","[{""",[Michael Chapman]
1071,7737,Resident Evil: Extinction,"[Years, after, the, Racoon, City, catastrophe,...","[Horror, Action, Science Fiction]","[clone, mutant, post-apocalyptic, dystopia, co...","[{""",[Russell Mulcahy]
1069,7453,The Hitchhiker's Guide to the Galaxy,"[Mere, seconds, before, the, Earth, is, to, be...","[Adventure, Comedy, Family, Science Fiction]","[bureaucracy, england, dolphin, android, based...","[{""",[Garth Jennings]


In [61]:
#now we aplly a transformation where we remove the spaces between each string from genres , keywords , cast and crew 
#we are doing to avoid the confusion for our model because when we will make tags each word will be converted to tags so if there are 2 same tags our model will be confused 
#example - if we want to watch a movie for sam worthingtoh but our model recommends movie for sam mendes beacause there are 2 same tags of sam so to avoid this confusion we are removing spaces from string so that it becomes a big tag , this same we apply on all the columns

In [67]:
def collapse(L):
    L1 = []
    for i in L:
        L1.append(i.replace(" ",""))
    return L1

In [69]:
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)
movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)

In [71]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[[, {, ""]",[JamesCameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[[, {, ""]",[GoreVerbinski]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[[, {, ""]",[SamMendes]
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[[, {, ""]",[ChristopherNolan]
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[[, {, ""]",[AndrewStanton]


In [73]:
#now we add all these 5 columns into asingle column naming tags we do concatenation of coulmns here
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [75]:
new = movies.drop(columns=['overview','genres','keywords','cast','crew'])

In [77]:
new.head()

,movie_id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


In [79]:
#now in tags column we covert all the list into strings
new['tags'] = new['tags'].apply(lambda x: " ".join(x))
new.head()

,movie_id,title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...
4,49529,John Carter,"John Carter is a war-weary, former military ca..."


In [81]:
new['tags'] = new['tags'].apply(lambda x:x.lower())

In [83]:
new.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [115]:
import nltk #now we apply stemming where it is a technique to remove the similar words and replace it with some common words example - dance,danced,dancing

In [117]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

In [119]:
def stem(text):
    y = []

    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)

In [121]:
new['tags'] = new['tags'].apply(stem)

In [123]:
#now we do text vectorization
new['tags'][0]

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powerrel mindandsoul 3d [ { " jamescameron'

In [125]:
new['tags'][1]

'captain barbossa, long believ to be dead, ha come back to life and is head to the edg of the earth with will turner and elizabeth swann. but noth is quit as it seems. adventur fantasi action ocean drugabus exoticisland eastindiatradingcompani loveofone\'slif traitor shipwreck strongwoman ship allianc calypso afterlif fighter pirat swashbuckl aftercreditssting [ { " goreverbinski'

In [127]:
#now our tasks is to find similarity between the tags so that when user enter the movie we can recommend him 5 such similar movies
#now we do vectorization where we convert ech tag in a movie into a vector , now each movie will be a vector when the user ask for similar movies the system will recommend him the vectors closest to the main movie
#the technique we use here is bag of words for text vectorization
#in this text vectorization hum stopwords ko nikal denge unko hum humare movies vector mai consider nhi karenge 

from sklearn.feature_extraction.text import CountVectorizer #this converts the tags column into numeric vectors 
cv = CountVectorizer(max_features=5000, stop_words='english') #limit is 5000 and remove the stopwords like is , the etc

vectors = cv.fit_transform(new['tags']).toarray()
#convert the sparse matrix into numpy array

In [128]:
vectors.shape

(4806, 5000)

In [131]:
vectors[0]

array([0, 0, 0, ..., 0, 0, 0], dtype=int64)

In [145]:
cv.get_feature_names_out() #we combined all the tags into one array

array(['000', '007', '10', ..., 'zombies', 'zone', 'zoo'], dtype=object)

In [147]:
#hume har movie ka dusre movie ke saath distance calculate karna hai . here,we do not calculate euclidean distance . here we will calculate cosine distance
#where we find the angle between to vectors(lines) .if the angle is 0 then it will be same vector and if it is grater than 0 then it will show some changes 
#for higher dimensional space we do not use euclidean distance and so to replace this we cosine distance
#diastance is inversely proportional to similarity. jitna do vectors ke bich ka theta small hai utne woh similar hai ,but agar theta bada hai toh woh movies dissimilar hai
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)

In [149]:
similarity.shape

(4806, 4806)

In [157]:
similarity

array([[1.        , 0.08858079, 0.09004503, ..., 0.04559608, 0.        ,
        0.        ],
       [0.08858079, 1.        , 0.06558258, ..., 0.02490677, 0.        ,
        0.0277137 ],
       [0.09004503, 0.06558258, 1.        , ..., 0.02531848, 0.        ,
        0.        ],
       ...,
       [0.04559608, 0.02490677, 0.02531848, ..., 1.        , 0.04003204,
        0.04279605],
       [0.        , 0.        , 0.        , ..., 0.04003204, 1.        ,
        0.08908708],
       [0.        , 0.0277137 , 0.        , ..., 0.04279605, 0.08908708,
        1.        ]])

In [159]:
def recommend(movie):
    
    movie_index = new[new['title'] == movie].index[0]
    
    distances = similarity[movie_index]
    
    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]
    
    for i in movies_list:
        print(new.iloc[i[0]].title)

In [161]:
recommend('The Dark Knight')

The Dark Knight Rises
Batman Begins
Batman Returns
Batman Forever
Batman


In [163]:
import pickle

In [165]:
pickle.dump(new,open('movie_list.pkl','wb'))
pickle.dump(similarity,open('similarity.pkl','wb'))

In [171]:
import os

os.chdir(r"C:\Users\Yashasvi\ml projects\movie-recommender-system")

print(os.getcwd())

C:\Users\Yashasvi\ml projects\movie-recommender-system


In [173]:
with open(".gitignore", "w") as file:
    file.write(""".env
__pycache__/
*.pyc
""")

In [175]:
import os

print(os.path.exists(".gitignore"))

True


In [177]:
with open(".env", "w") as file:
    file.write("OMDB_API_KEY=2cbd66b9")

In [179]:
import os

print(os.path.exists(".env"))

True
